In [1]:
import os

import polars as pl

import iqplot

import icutils

import bokeh.io
bokeh.io.output_notebook()

Loading BokehJS ...

Here, we demonstrate some example of computing grades for Integrated Core. We will use a dummy data set of grades that are stored locally. 

In [2]:
gradesheet_path = 'gradesheets/'

gradesheet = os.path.join(gradesheet_path, 'IC_dummy_gradesheet - gradesheet.csv')
late_multiplier = os.path.join(gradesheet_path, 'IC_dummy_gradesheet - late_multiplier.csv')
engagement = os.path.join(gradesheet_path, 'IC_dummy_gradesheet - engagement.csv')

The key functionality of the `icgrade` package is to wrangle the CVS files into a tidy data frame.

In [3]:
# Load in spreadsheets
df = icutils.wrangle(gradesheet, late_multiplier, engagement)

# Take a look
df.head()

term,assignment,assignment_type,problem,points,counts toward grade,subject 1,subject 2,subject 3,lab,due date,grader,student,score,late_multiplier
str,str,str,i64,i64,bool,str,str,str,bool,date,str,str,f64,f64
"""fall""","""1a""","""homework""",0,0,false,null,null,null,false,2025-10-01,"""dos santos, marc""","""Amaya, Frankie""",3.4,1.0
"""fall""","""1a""","""homework""",1,9,true,"""geology""",null,null,false,2025-10-01,"""razov, ante""","""Amaya, Frankie""",6.5,1.0
"""fall""","""1a""","""homework""",2,33,true,"""chemistry""",null,null,false,2025-10-01,"""thorrington, john""","""Amaya, Frankie""",22.5,1.0
"""fall""","""1a""","""homework""",3,19,true,"""geology""",null,null,false,2025-10-01,"""nikolav, oka""","""Amaya, Frankie""",18.5,1.0
"""fall""","""1a""","""homework""",4,38,true,"""geology""","""biology""",null,false,2025-10-01,"""razov, ante""","""Amaya, Frankie""",31.5,1.0


This tall tidy format is well-suited for slicing and dicing the data set to compute all kinds of things. Below are some examples.

## Make a report of scores from winter midterm exam c

`icutils` has a report generator for given assignment. The result is written to an HTML file.

In [4]:
icutils.assignment_dashboard(df, 'winter', 'midterm2c')

Dashboard saved to winter_midterm2c_scores.html.


## Physics grades for spring term

Below is a calculation of the physics scores for the spring term with the respective components of the course counted with their weights. Note that the research funding proposal and the engagement scores count toward all subjects and are included here.

In [5]:
df_ph_spring = icutils.subject_scores(
    df,
    subject='physics',
    term='spring',
    homework_weight=35,
    midterm_weight=20,
    final_weight=25,
    rfp_weight=10,
    engagement_weight=10,
)

# Output to CSV if you want to edit as a spreadsheet
df_ph_spring.write_csv('ic_physics_score_summary_spring.csv')

# Take a look
df_ph_spring

student,homework,midterm,final,rfp,engagement,total
str,f64,f64,f64,f64,f64,f64
"""Amaya, Frankie""",0.782634,0.706186,0.783333,1.0,1.0,0.810992
"""Bouanga, Denis""",0.875,0.948454,0.9,1.0,0.8,0.900941
"""Choiniere, Mathieu""",0.666667,0.731959,0.761111,1.0,1.0,0.770003
"""Delgado, Mark""",0.791958,0.835052,0.816667,1.0,1.0,0.848362
"""Ebobisse, Jeremy""",0.785548,0.706186,0.783333,0.8,1.0,0.792012
…,…,…,…,…,…,…
"""Smolyakov, Ardem""",0.755245,0.757732,0.777778,1.0,0.65,0.775327
"""Son, Heung Min""",0.93007,0.984536,0.955556,1.0,1.0,0.961321
"""Tafari, Nkosi""",0.788462,0.731959,0.8,1.0,1.0,0.822353


## ECDF of cumulative final exam scores for the fall term

In [6]:
df_final = (
    df
    .filter((pl.col("term") == "fall") & (pl.col('assignment_type') == 'final'))
    .group_by(["student"]).agg(
        (pl.col("score").sum() / pl.col("points").sum()).alias('score')
    )
    .sort(by='student')
)

bokeh.io.show(
    iqplot.ecdf(
        df_final,
        q='score',
        tooltips=[('name', '@student')],
    )
)

## A plot of time spent on homework

In [7]:
# Example to create plot of time spent on homework
df = icutils.wrangle(
    gradesheet, late_multiplier, engagement, homework_nulls_to_zero=False
)
df_time = df.filter(pl.col("problem") == 0)[
    ["term", "assignment", "student", "score"]
].drop_nulls()

p = iqplot.strip(
    df_time,
    q="score",
    cats=["term", "assignment"],
    tooltips=[("", "@student")],
    spread="jitter",
    x_axis_label="time spent on assignment (hr)",
    order=[
        (term, assignment)
        for term in ["fall", "winter", "spring"]
        for assignment in sorted(list(df_time["assignment"].unique()))
    ],
    frame_height=1200,
)

bokeh.io.show(p)